In [5]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from collections import deque

plt.rcParams['axes.titlesize'] = 16   
plt.rcParams['axes.labelsize'] = 14  
plt.rcParams['xtick.labelsize'] = 12  
plt.rcParams['ytick.labelsize'] = 12  
plt.rcParams['legend.fontsize'] = 12 

class TrajectoryScoring:

    def __init__(self, gt_dict, pred_dict, save_vis_path):
        self.gt_dict = gt_dict
        self.pred_dict = pred_dict
        self.save_vis_path = save_vis_path
        self.driving_style = 'conservative' # aggressive or conservative
        self.driving_scene = 'city' # city or highway
        self.speed_history = deque(maxlen=12)  # Store the last average speeds
        self.b_spline_path = os.path.join(self.save_vis_path, 'b_spline')
        self.speed_range = (None, None) 
        self.sample = {}
        self.speed_lower_bound = -0.1
        self.speed_upper_bound = 40.0
        self.default_city_road_speed_limit = 15.67
        self.default_highway_speed_limit = 29.06
        self.best_trajectory_queue = deque(maxlen=3)  # 保存最佳轨迹的队列，最多3条
        
    def ensure_directory_exists(self, path):
        """Ensure the directory exists, and if not, create it."""
        if not os.path.exists(path):
            os.makedirs(path)
            print(f"Directory created: {path}")
        else:
            pass

    def transform_data(self):
        agent_bboxes = self.pred_dict["agent_bboxes"]  # N x 5, x, y, w, h, yaw
        agent_traj = self.pred_dict["agent_traj"]  # N x T x 2
        map_pts = self.pred_dict["map_pts"]  # N x 20 x 2
        map_labels = self.pred_dict["map_labels"]  # N
        pred_ego_traj = self.pred_dict["ego_traj"]  # T x 2
        gt_ego_traj = self.gt_dict["ego_traj"]
        target_point = self.pred_dict["target_point"]
        alpha = self.pred_dict["alpha"]

        agent_num = agent_bboxes.shape[0]
        agent_bbox = agent_bboxes[:, :4]
        agent_yaw = agent_bboxes[:, 4]
        agent_trajreg = agent_traj[..., [1, 0]]
        agent_trajreg[..., 0] = -agent_trajreg[..., 0]
       
        if agent_num == 0:
            agent_trajreg = np.zeros((0, 60))
        else:
            agent_trajreg = agent_trajreg.reshape(agent_num, -1)
        agent_mask = np.ones((agent_num, 30))

        agent_bbox_vcs = agent_bbox
        agent_bbox_vcs = agent_bbox_vcs[..., [1, 0, 2, 3]]
        agent_bbox_vcs[..., 0] = -agent_bbox_vcs[..., 0]

        vcs_yaw = agent_yaw[:, None]

        self.sample["gt_attr_labels"] = np.concatenate([agent_trajreg, agent_mask, agent_bbox_vcs[:, :2], vcs_yaw, agent_bbox_vcs[:, 2:], ], axis=1)
        self.sample["gt_labels_3d"] = np.ones(agent_num)

        # odo 处理
        pred_vcs_odo_info = pred_ego_traj
        pred_vcs_odo_info = pred_vcs_odo_info[..., [1, 0]]
        pred_vcs_odo_info[..., 0] = -pred_vcs_odo_info[..., 0]
        self.sample["pred_ego_fut_trajs"] = pred_vcs_odo_info

        new_anchors = self.pred_dict["anchors"]
        new_anchors = new_anchors[..., [1, 0]]
        new_anchors[..., 0] = -new_anchors[..., 0]
        self.sample["anchors"] = new_anchors.cpu().numpy()

        self.sample["alpha"] = (alpha - alpha.min()) / alpha.max()

        gt_vcs_odo_info = gt_ego_traj
        gt_vcs_odo_info = gt_vcs_odo_info[..., [1, 0]]
        gt_vcs_odo_info[..., 0] = -gt_vcs_odo_info[..., 0]
        self.sample["gt_ego_fut_trajs"] = gt_vcs_odo_info

        # map 计算
        lines = map_pts
        labels = map_labels

        self.sample["map_gt_bboxes_3d"] = lines
        self.sample["map_gt_labels_3d"] = labels

        target_point = target_point[:, [1, 0]]
        target_point[:, 0] = -target_point[:, 0]
        self.sample["target_info"] = target_point[0]

        self.sample["expert_idxs"] = self.pred_dict["expert_idxs"]
        self.sample["post_process_idxs"] = self.pred_dict["post_process_idxs"]

    def calculate_distance_to_target(self, trajectory, target_point):
        return np.linalg.norm(trajectory[-1] - target_point)

    def check_collision(self, trajectory, other_vehicles_bboxes):
        """
        Check if the trajectory collides with any bounding boxes of other vehicles.
        :param trajectory: Array of shape (T, 2), where T is the number of timesteps.
        :param other_vehicles_bboxes: List of bounding boxes, each defined as [x_center, y_center, width, height, yaw].
        :return: True if collision detected, False otherwise.
        """
        for bbox in other_vehicles_bboxes:
            x_center, y_center, width, height, yaw = bbox
            # Create bounds for the bounding box
            x_min = x_center - width / 2
            x_max = x_center + width / 2
            y_min = y_center - height / 2
            y_max = y_center + height / 2

            # Check each point in the trajectory
            for point in trajectory:
                x, y = point
                if x_min <= x <= x_max and y_min <= y <= y_max:
                    return True  # Collision detected
        return False  # No collision detected
    
    def calculate_collisions_with_agents(self):
        trajectories = self.sample['pred_ego_fut_trajs']
        agent_boxes = self.sample['gt_attr_labels']  # Assuming this contains agent box info
        agent_labels = self.sample['gt_labels_3d']  # Labels for the agents
        collision_scores = []
        for traj in trajectories:
            collision = 0
            for agent, label in zip(agent_boxes, agent_labels):
                if label in [0, 1, 2, 3]:  # Assuming these labels are for relevant agents
                    # Convert agent box format to [x_center, y_center, width, height, yaw]
                    bbox = [agent[0], agent[1], agent[3], agent[4], agent[2]]
                    if self.check_collision(traj, [bbox]):  # Check collision with each agent
                        collision = 1
                        break
            collision_scores.append(collision)
        return collision_scores

    def calculate_deviation_from_lanes(self):
        trajectories = self.sample['pred_ego_fut_trajs']
        lane_lines = self.sample['map_gt_bboxes_3d']
        lane_labels = self.sample['map_gt_labels_3d']
        deviation_scores = []
        threshold = 1.0  # 设置阈值

        for traj in trajectories:
            min_deviation = float('inf')  # 初始化最小偏离值为无穷大
            for line, label in zip(lane_lines, lane_labels):
                if label in [0, 1, 3]:  # 只考虑相关的车道线标签
                    # 计算轨迹与当前车道线的最小距离
                    deviation = np.min(np.linalg.norm(traj[:, None, :] - line, axis=2))
                    # 更新最小偏离值
                    min_deviation = min(min_deviation, deviation)
            
            # 如果最小偏离值小于阈值，则计算成本，否则成本为0
            deviation_cost = 0 if min_deviation >= threshold else min_deviation
            deviation_scores.append(deviation_cost)
        return deviation_scores

    def calculate_average_speeds(self, trajectories):
        """
        Calculate average speeds for each trajectory.
        :param trajectories: Array of shape (num_trajectories, num_points, 2)
        :return: Array of average speeds of shape (num_trajectories,)
        """
        speeds = self.calculate_speeds(trajectories)
        average_speeds = np.mean(speeds, axis=1)
        return average_speeds

    def calculate_speeds(self, trajectories):
        """
        Calculate speeds for each point in each trajectory.
        :param trajectories: Array of shape (num_trajectories, num_points, 2)
        :return: Array of speeds of shape (num_trajectories, num_points-1)
        """
        speeds = np.linalg.norm(np.diff(trajectories, axis=1), axis=2)
        return speeds

    def calculate_dynamics(self, trajectory):
        """
        Calculate longitudinal and lateral velocities, accelerations, and jerks for a trajectory.
        """
        dt = 0.1  # assuming trajectory points are at 0.1 second intervals
        velocities = np.diff(trajectory, axis=0) / dt
        long_velocities = velocities[:, 0]  # Assuming x is the longitudinal direction
        lat_velocities = velocities[:, 1]  # Assuming y is the lateral direction

        long_accelerations = np.diff(long_velocities) / dt
        lat_accelerations = np.diff(lat_velocities) / dt

        long_jerks = np.diff(long_accelerations) / dt
        lat_jerks = np.diff(lat_accelerations) / dt

        return long_velocities, lat_velocities, long_accelerations, lat_accelerations, long_jerks, lat_jerks
    
    def lat_comfort_cost(self, long_velocities, lat_velocities, long_accelerations, lat_jerks):
        max_cost = 0.0
        for i in range(len(long_velocities)):
            s_dot = long_velocities[i]
            s_dotdot = long_accelerations[i] if i < len(long_accelerations) else 0.0
            l_prime = lat_velocities[i]
            l_primeprime = lat_jerks[i] if i < len(lat_jerks) else 0.0
            cost = l_primeprime * s_dot * s_dot + l_prime * s_dotdot
            max_cost = max(max_cost, abs(cost))
        return max_cost

    def lon_comfort_cost(self, long_jerks):
        cost_sqr_sum = 0.0
        cost_abs_sum = 0.0
        longitudinal_jerk_upper_bound = 5.0  
        numerical_epsilon = 1e-6  
        for jerk in long_jerks:
            cost = jerk / longitudinal_jerk_upper_bound
            cost_sqr_sum += cost * cost
            cost_abs_sum += abs(cost)
        return cost_sqr_sum / (cost_abs_sum + numerical_epsilon)

    def calculate_speed_cost(self, average_speeds, speed_range):
        # 根据场景设置速度限制
        if self.driving_scene == 'city':
            speed_limit = self.default_city_road_speed_limit
        elif self.driving_scene == 'highway':
            speed_limit = self.default_highway_speed_limit
        else:
            raise ValueError("Invalid driving scene type. Must be 'city' or 'highway'.")

        speed_costs = []
        for speed in average_speeds:
            # 如果速度超过限制，设置无穷大的成本
            if speed > speed_limit:
                speed_cost = float('inf')
            else:
                # 在速度范围内计算成本
                if self.driving_style == 'aggressive':
                    # 激进模式下，速度慢的轨迹施加成本
                    if speed < speed_range[0]:
                        speed_cost = (speed_range[0] - speed) / speed_range[0]
                    else:
                        speed_cost = 0
                elif self.driving_style == 'conservative':
                    # 保守模式下，速度快的轨迹施加成本
                    if speed > speed_range[1]:
                        speed_cost = (speed - speed_range[1]) / speed_range[1]
                    else:
                        speed_cost = 0
            speed_costs.append(speed_cost)

        return speed_costs


    def calculate_centripetal_acceleration_cost(self, trajectory, speeds):
        """
        Calculate the centripetal acceleration cost for a single trajectory.
        :param trajectory: Array of shape (num_points, 2) containing a single trajectory.
        :param speeds: Array of speeds of shape (num_points-1,) for the trajectory.
        :return: Centripetal acceleration cost for the trajectory.
        """
        centripetal_acc_sum = 0.0
        centripetal_acc_sqr_sum = 0.0
        numerical_epsilon = 1e-6  # 避免除以零

        # 计算曲率
        curvatures = [self.calculate_curvature(trajectory[i - 1], trajectory[i], trajectory[i + 1])
                      for i in range(1, len(trajectory) - 1)]

        # 计算向心加速度和成本
        for i, curvature in enumerate(curvatures):
            centripetal_acc = speeds[i] ** 2 * curvature
            centripetal_acc_sum += np.fabs(centripetal_acc)
            centripetal_acc_sqr_sum += centripetal_acc ** 2

        cost = centripetal_acc_sqr_sum / (centripetal_acc_sum + numerical_epsilon)
        return cost

    def calculate_curvature(self, p1, p2, p3):
        """
        Calculate the curvature given three consecutive points on the trajectory.
        :param p1, p2, p3: Consecutive points on the trajectory.
        :return: Curvature.
        """
        # 使用三点计算曲率的方法
        k = np.linalg.norm(np.cross(p2 - p1, p3 - p2)) / np.linalg.norm(p2 - p1)**2
        return k

    def compute_scores(self):
        weights = {
            'target_distance': 2.0,
            'collision': 6.0,
            'lane_deviation': 1.0,
            'speed': 2.0,  
            'lat_comfort': 3.0,
            'lon_comfort': 3.0,
            'centripetal_acceleration': 3.5,  
        }

        # 获取预测轨迹并切片以保留每条轨迹的前12个点
        pred_trajectories = self.sample['pred_ego_fut_trajs'][:, :12, :]
        target_point = self.sample['target_info']

        # 如果队列中有上一帧的最佳轨迹，将它们添加到当前帧的轨迹列表中
        if self.best_trajectory_queue:
            previous_best_trajectories = np.array(self.best_trajectory_queue)
            pred_trajectories = np.concatenate([previous_best_trajectories, pred_trajectories], axis=0)
        
        # Calculate average speeds for each trajectory
        average_speeds = self.calculate_average_speeds(pred_trajectories)

        # Update the speed range based on historical average speeds
        if self.speed_history:
            historical_avg_speed = np.mean(self.speed_history)
            self.speed_range = (0.8 * historical_avg_speed, 1.2 * historical_avg_speed)
        else:
            # If there's no history yet, use the current average speeds
            current_avg_speed = np.mean(average_speeds)
            self.speed_range = (0.8 * current_avg_speed, 1.2 * current_avg_speed)

        scores = []
        for i, pred_traj in enumerate(pred_trajectories):
            target_distance = self.calculate_distance_to_target(pred_traj, target_point)
            collision = self.calculate_collisions_with_agents()[i]
            deviation = self.calculate_deviation_from_lanes()[i]

            # Calculate speed cost for the current trajectory
            speed_cost = self.calculate_speed_cost([average_speeds[i]], self.speed_range)[0]

            # Calculate dynamics for the trajectory
            long_velocities, lat_velocities, long_accelerations, lat_accelerations, long_jerks, lat_jerks = self.calculate_dynamics(pred_traj)

            # Calculate comfort costs
            lat_comfort = self.lat_comfort_cost(long_velocities, lat_velocities, long_accelerations, lat_jerks)
            lon_comfort = self.lon_comfort_cost(long_jerks)

            # Calculate centripetal acceleration cost for the current trajectory
            speeds = self.calculate_speeds(np.expand_dims(pred_traj, axis=0))[0]
            centripetal_acceleration_cost = self.calculate_centripetal_acceleration_cost(pred_traj, speeds)

            # Calculate total score
            total_score = (weights['target_distance'] * target_distance +
                        weights['collision'] * collision +
                        weights['lane_deviation'] * deviation +
                        weights['speed'] * speed_cost + 
                        weights['lat_comfort'] * lat_comfort + 
                        weights['lon_comfort'] * lon_comfort + 
                        weights['centripetal_acceleration'] * centripetal_acceleration_cost)
            
            scores.append(total_score)

        # Update the historical average speed
        self.speed_history.append(np.mean(average_speeds))
        # 在得分计算结束后，找到成本最小的轨迹并更新队列
        min_score_idx = np.argmin(scores[len(self.best_trajectory_queue):])  # 忽略队列中的轨迹
        self.best_trajectory_queue.append(pred_trajectories[min_score_idx])

        return scores
    
    def visualize_results(self, scores, timestamp):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 10))  # 创建两个子图

        # 设置显示范围
        ax1.set_xlim(-20, 20)
        ax1.set_ylim(-30, 30)
        ax2.set_xlim(-20, 20)
        ax2.set_ylim(-30, 30)

        # 绘制基本的map, 自车, 目标点
        self.plot_map_and_ego(ax1)
        self.plot_map_and_ego(ax2)

        # 左子图：绘制所有预测的轨迹和分数
        color_map = plt.cm.get_cmap('viridis', len(scores))
        for i, (traj, score) in enumerate(zip(self.sample["pred_ego_fut_trajs"], scores)):
            ax1.plot(traj[:, 0], traj[:, 1], marker='o', color=color_map(i), label=f'Score: {score:.2f}', linewidth=1)

        # 右子图：绘制成本最低和最高的轨迹，以及GT轨迹
        # 突出显示成本最低（得分最低）的轨迹
        min_score_idx = np.argmin(scores)
        min_traj = self.sample["pred_ego_fut_trajs"][min_score_idx][:12]
        ax2.plot(min_traj[:, 0], min_traj[:, 1], 'r-', linewidth=5, label='Lowest Cost Trajectory')

        # 突出显示成本最高（得分最高）的轨迹
        # max_score_idx = np.argmax(scores)
        # max_traj = self.sample["pred_ego_fut_trajs"][max_score_idx][:12]
        # ax2.plot(max_traj[:, 0], max_traj[:, 1], 'b-', linewidth=3, label='Highest Cost Trajectory')

        # 绘制GT轨迹
        gt_ego_fut_trajs = self.sample["gt_ego_fut_trajs"]
        ax1.plot(gt_ego_fut_trajs[:, 0], gt_ego_fut_trajs[:, 1], 'g-', label='Ground Truth Ego Trajectory')
        ax2.plot(gt_ego_fut_trajs[:, 0], gt_ego_fut_trajs[:, 1], 'g-', label='Ground Truth Ego Trajectory')

        # 绘制目标点
        target_point = self.sample['target_info']
        ax1.plot(target_point[0], target_point[1], 'g*', markersize=15, label='Target Point')
        ax2.plot(target_point[0], target_point[1], 'g*', markersize=15, label='Target Point')

        # 设置图例和标题
        ax1.legend(loc='upper right')
        ax1.set_title('Trajectory Visualizations with Scores')
        ax2.legend(loc='upper right')
        ax2.set_title('Best and Worst Trajectory Visualizations')

        # 设置坐标轴标签
        ax1.set_xlabel('X Coordinate')
        ax1.set_ylabel('Y Coordinate')
        ax2.set_xlabel('X Coordinate')
        ax2.set_ylabel('Y Coordinate')

        # 关闭网格
        ax1.grid(False)
        ax2.grid(False)

        # 保存图像
        plt.savefig(os.path.join(self.b_spline_path, f"{timestamp}.png"))
        plt.close()

    def plot_map_and_ego(self, ax):
        
        yaw, w, l = np.pi / 2, 1.5, 4.0
        ego_corner = np.array(
            [
                [-0.5 * l, 0.5 * w],
                [0.5 * l, 0.5 * w],
                [0.5 * l, -0.5 * w],
                [-0.5 * l, -0.5 * w],
                [-0.5 * l, 0.5 * w],
            ]
        ).T
        ego_corner = np.matmul(
            np.array(
                [[math.cos(yaw), -math.sin(yaw)], [math.sin(yaw), math.cos(yaw)]]
            ),
            ego_corner,
        )  # (2,5)

        # plot agent box
        attr_label = self.sample["gt_attr_labels"]
        gt_agent_lcf_feat = attr_label[:, 90:95]
        gt_agent_label = self.sample["gt_labels_3d"]
        cls2color = {0: "r", 1: "g", 2: "y", 3: "m"}
        for a in range(gt_agent_lcf_feat.shape[0]):
            xy, yaw, w, l = (
                gt_agent_lcf_feat[a, :2],
                gt_agent_lcf_feat[a, 2].item(),
                gt_agent_lcf_feat[a, 3].item(),
                gt_agent_lcf_feat[a, 4].item(),
            )
            color = cls2color[gt_agent_label[a]]
            corner = np.array(
                [
                    [-0.5 * l, 0.5 * w],
                    [0.5 * l, 0.5 * w],
                    [0.5 * l, -0.5 * w],
                    [-0.5 * l, -0.5 * w],
                    [-0.5 * l, 0.5 * w],
                ]
            ).T
            corner = (
                np.matmul(
                    np.array(
                        [
                            [math.cos(yaw), -math.sin(yaw)],
                            [math.sin(yaw), math.cos(yaw)],
                        ]
                    ),
                    corner,
                )
                + xy[None].T
            )  # (2,5)
            ax.plot(
                corner[0, :],
                corner[1, :],
                color,
                linewidth=0.5,
                alpha=0.8,
                zorder=-1,
            )

        ### plot ego box
        
        ax.plot(
            ego_corner[0, :],
            ego_corner[1, :],
            "b",
            linewidth=0.5,
            alpha=0.8,
            zorder=-1,
        )

        # 绘制车道线和地图标签
        map2color = {
            0: "y",  # 'solid_lane'
            1: "red",  # 'roadedge'
            2: "orange",  # 'crosswalk'
            3: "y",  # centerline
            -1: "black",  # 'others'
        }
        gt_map_pts = self.sample["map_gt_bboxes_3d"]  # [N,20,2]
        gt_map_labels_3d = self.sample["map_gt_labels_3d"]
        for gt_pts, gt_label in zip(gt_map_pts, gt_map_labels_3d):
            pts = gt_pts
            cls = gt_label
            x = np.array([pt[0] for pt in pts])
            y = np.array([pt[1] for pt in pts])
            ax.plot(x, y, color=map2color[cls], linewidth=1, alpha=0.8, zorder=-1, linestyle="--")

    def run(self, timestamp):
        self.ensure_directory_exists(self.b_spline_path)
        self.transform_data()
        score = self.compute_scores()
        self.visualize_results(score,timestamp)
    
   


In [ ]:
gt_dict = {

}

pred_dict ={

}
